## ReelRank: controlled comparisons

This notebook reads measured results, not target numbers. Models use training-fitted features and standard-validation selection. The random log was inspected in previous development and is explicitly exploratory. All tables below were executed by `python scripts/build_analysis_notebook.py`.

In [1]:
import json
from pathlib import Path
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'reelrank' / 'ranking').exists())
def load(name):
    return json.loads((root / 'results' / f'{name}.json').read_text())
print('Source root:', root)

Source root: /Users/priyadharshan/Desktop/reelrank


## Ranking: choose on validation

Same feature information and samples across models. LR is selected by standard-validation long-view GAUC. DeepFM has the best validation long-view log loss; MMoE is retained as an experiment. These are single-seed results, not evidence that one architecture universally wins.

In [2]:
rows = []
for model in ['lr', 'deepfm', 'mmoe']:
    for suffix in ['_validation', '']:
        record = load(f'ranking_{model}{suffix}')
        m = record['metrics']['long_view']
        rows.append({'model': model, 'split': record['config']['evaluation_split'], **m})
print(pd.DataFrame(rows).to_string(index=False))
print('MMoE validation learning curve:')
print(pd.DataFrame(load('ranking_mmoe_validation')['extra']['training_history']).to_string(index=False))

 model                       split      auc     gauc  logloss
    lr         standard_validation 0.729675 0.648446 0.552508
    lr random_exposure_exploratory 0.655899 0.596268 0.445948
deepfm         standard_validation 0.728214 0.643895 0.549303
deepfm random_exposure_exploratory 0.658188 0.598931 0.410318
  mmoe         standard_validation 0.673887 0.617256 0.579855
  mmoe random_exposure_exploratory 0.592375 0.585175 0.372126
MMoE validation learning curve:
 epoch  train_loss  validation_logloss
     1    0.483229            0.463025
     2    0.464854            0.459433
     3    0.460004            0.456343


## Retrieval on matched users

All methods use long-view relevance, the same seed-42 2,000-user cohort and allow repeats. Earlier results used different cohorts and policies, so before/after changes cannot be attributed to one fix.

In [3]:
rows = []
for method in ['popularity', 'itemcf', 'two_tower']:
    record = load(f'retrieval_{method}')
    rows.append({'method': method, **record['metrics']})
print(pd.DataFrame(rows).to_string(index=False))

    method  coverage  hit_rate@10  hit_rate@200  hit_rate@50  recall@10  recall@200  recall@50
popularity  0.026375       0.0185        0.2395       0.0875   0.003754    0.058418   0.017233
    itemcf  0.881973       0.0075        0.2220       0.0660   0.001418    0.053501   0.012626
 two_tower  0.340630       0.0240        0.2560       0.0885   0.004214    0.069556   0.019709


## Cascade and uncertainty

Both candidate generators receive the same safety policy. Full-pool NDCG is overlap with sparse observed positives, not an estimate of online benefit. Paired bootstrap intervals quantify user variation only. An interval crossing zero does not establish a positive effect.

In [4]:
rows = []
for model in ['lr', 'mmoe']:
    for safety in ['no_safety', 'with_safety']:
        r = load(f'e2e_{model}_{safety}')
        m = r['metrics']; ci = m['paired_bootstrap']
        rows.append({'ranker': model, 'safety': safety, 'users': m['n_users'],
                     'two_tower_ndcg': m['ndcg@10_two_tower'], 'popularity_ndcg': m['ndcg@10_popularity'],
                     'relative_lift': m.get('lift_vs_pop'), 'difference': ci['mean_difference'],
                     'difference_ci95': ci['difference_ci95'],
                     'two_tower_hit_users': m['users_with_positive_hit_two_tower'],
                     'popularity_hit_users': m['users_with_positive_hit_popularity']})
print(pd.DataFrame(rows).to_string(index=False))

ranker      safety  users  two_tower_ndcg  popularity_ndcg  relative_lift  difference       difference_ci95  two_tower_hit_users  popularity_hit_users
    lr   no_safety   2000        0.003532         0.002587       0.365630    0.000946 [-0.000109, 0.002053]                   48                    43
    lr with_safety   2000        0.003587         0.002790       0.285416    0.000796 [-0.000265, 0.001892]                   48                    45
  mmoe   no_safety   2000        0.002969         0.002548       0.165037    0.000421  [-0.000697, 0.00145]                   41                    39
  mmoe with_safety   2000        0.003007         0.002643       0.137801    0.000364  [-0.000793, 0.00144]                   42                    38


## Index choice

Choose based on this measured candidate-pool size. Timing noise and hardware matter; these index benchmarks are not end-to-end request latency.

In [5]:
a = load('retrieval_faiss')['metrics']
print(pd.DataFrame([{'index': key, **a[key]} for key in ['flat', 'hnsw']]).to_string(index=False))
print('HNSW recall vs exact:', a['hnsw_recall_vs_flat'])

index  batch_mean_ms  mean_ms   p50_ms   p99_ms
 flat       0.077911 0.099997 0.082333 0.394045
 hnsw       0.052402 0.062262 0.055688 0.171186
HNSW recall vs exact: 0.99182


## Safety limitations

Caption-only weak-label evaluation. All-safe human review cannot establish unsafe recall. The uncertain-band count is hypothetical routing, not measured LLM serving. Existing labels were not regenerated by the correctness fixes.

In [6]:
for name in ['safety_classifier', 'safety_human_kappa']:
    print(name, json.dumps(load(name)['metrics'], indent=2))

safety_classifier {
  "auc": 0.6511142061281338,
  "corr_safety_score_vs_hate_rate": -0.022296500113094805,
  "f1": 0.10810810810810811,
  "n_test": 375,
  "n_train": 1125,
  "precision": 0.09523809523809523,
  "recall": 0.125,
  "unsafe_rate_test": 0.042666666666666665
}
safety_human_kappa {
  "agreement": 0.97,
  "cohen_kappa": 0.0,
  "confusion_matrix": {
    "labels": [
      "safe",
      "unsafe"
    ],
    "matrix": [
      [
        194,
        0
      ],
      [
        6,
        0
      ]
    ]
  },
  "human_unsafe": 0,
  "judge_unsafe": 6,
  "n_labeled": 200,
  "n_spotcheck_rows": 200,
  "n_unlabeled": 0
}


## Serving and reproducibility

Measured achieved throughput includes Locust think time. The source snapshot and hashes identify the implementation and artifacts; per-user cascade records permit independent uncertainty checks. No online experiment or production-scale claim is made.

In [7]:
print(json.dumps(load('serving_loadtest')['metrics'], indent=2))
for name in ['ranking_lr', 'retrieval_two_tower', 'e2e_lr_with_safety', 'serving_loadtest']:
    r = load(name)
    print(name, 'record:', r.get('record_path'), 'source:', r.get('provenance', {}).get('source_hash'))

{
  "concurrency": 8,
  "errors": 0,
  "hardware": {
    "platform": "macOS-26.5.1-arm64-arm-64bit-Mach-O",
    "processor": "arm",
    "python": "3.13.15"
  },
  "mean_ms": 36.91536277302164,
  "n_requests": 2890,
  "p50_ms": 31.0,
  "p95_ms": 79.0,
  "p99_ms": 100.0,
  "qps": 48.357230382080786,
  "tool": "locust"
}
ranking_lr record: results/runs/20261101T073349Z-ranking_lr-4a15ba26/result.json source: a858abbaab78
retrieval_two_tower record: results/runs/20261101T073313Z-retrieval_two_tower-6529ff58/result.json source: a858abbaab78
e2e_lr_with_safety record: results/runs/20261101T073615Z-e2e_lr_with_safety-149cd955/result.json source: 23afc75ce95e
serving_loadtest record: results/runs/20261101T073944Z-serving_loadtest-2ee00139/result.json source: 29e1ead06c48
